# XGBoost — Second-Order Gradients From Scratch

XGBoost (Chen & Guestrin, 2016) won more Kaggle competitions than any other
algorithm when it was released. This notebook shows exactly why it beats
vanilla Gradient Boosting.

**The two key innovations:**
1. Use both gradient (g) AND hessian (h) — a better quadratic approximation
2. Regularize the tree structure directly in the objective — L1/L2 on leaf weights


In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression, make_classification
from sklearn.model_selection import train_test_split
%matplotlib inline

from xgboost_scratch import XGBRegressor, XGBClassifier, SquaredErrorObjective, LogisticObjective
from gradient_boosting import GradientBoostingRegressor
np.random.seed(42)


## 1. First vs Second Order: What Does the Hessian Add?

Standard GB fits trees to: `r_i = y_i - F(x_i)` (first-order residuals)

XGBoost fits trees to minimize: `Σ [g_i * w + 0.5 * h_i * w²]`

This second-order Taylor expansion gives a better local approximation of the
true loss, leading to better split decisions.

For MSE: h_i = 1 (constant) → XGBoost ≈ GB
For log-loss: h_i = p_i(1-p_i) → near the decision boundary h is large,
far from it h → 0. This naturally weights uncertain samples more.


In [ ]:
# Visualize g and h for logistic loss
F_vals = np.linspace(-5, 5, 200)
p_vals = LogisticObjective.sigmoid(F_vals)
y_true = 1   # true label is 1

g = p_vals - y_true          # gradient
h = p_vals * (1 - p_vals)   # hessian

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(F_vals, p_vals, color='#4C72B0', linewidth=2.5)
axes[0].axhline(0.5, linestyle='--', color='gray', alpha=0.5)
axes[0].set_title('Predicted probability p = σ(F)', fontweight='bold')
axes[0].set_xlabel('F (log-odds)')

axes[1].plot(F_vals, g, color='#C44E52', linewidth=2.5)
axes[1].axhline(0, linestyle='--', color='gray', alpha=0.5)
axes[1].set_title('Gradient g = p - y  (first derivative)', fontweight='bold')
axes[1].set_xlabel('F (log-odds)')

axes[2].plot(F_vals, h, color='#55A868', linewidth=2.5)
axes[2].set_title('Hessian h = p(1-p)  (second derivative)', fontweight='bold')
axes[2].set_xlabel('F (log-odds)')
axes[2].annotate('Near boundary:
high uncertainty → large h', xy=(0, 0.25),
                  xytext=(1.5, 0.2), arrowprops=dict(arrowstyle='->'))

plt.suptitle('Logistic Loss: Gradient and Hessian', fontweight='bold', fontsize=12)
plt.tight_layout(); plt.show()
print("Key insight: h = p(1-p) = Bernoulli variance")
print("High h = uncertain prediction = needs more correction")
print("Low h  = confident prediction = less correction needed")


## 2. The XGBoost Gain Formula

Instead of impurity reduction, XGBoost uses:

```
Gain = ½ × [G_L²/(H_L+λ) + G_R²/(H_R+λ) - (G_L+G_R)²/(H_L+H_R+λ)] - γ

Optimal leaf weight: w* = -G / (H + λ)
```

The λ in the denominator **regularizes** leaf weights — leaves with low total
hessian (low sample count or low confidence) get shrunk toward zero automatically.


In [ ]:
# Show effect of lambda on leaf weights
G_vals = np.linspace(-10, 10, 100)
fig, ax = plt.subplots(figsize=(8, 4))
for lam in [0, 0.1, 1.0, 5.0, 10.0]:
    H = 10.0
    w_star = -G_vals / (H + lam)
    ax.plot(G_vals, w_star, linewidth=2, label=f'λ={lam}')
ax.axhline(0, color='black', linewidth=0.8, alpha=0.5)
ax.axvline(0, color='black', linewidth=0.8, alpha=0.5)
ax.set_xlabel('Sum of gradients G'); ax.set_ylabel('Optimal leaf weight w*')
ax.set_title('Effect of L2 regularization (λ) on leaf weights
'
             'Higher λ → weights shrunk toward 0', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()


## 3. XGBoost vs Vanilla GB: Regularization Effect

In [ ]:
X, y = make_regression(n_samples=500, n_features=20, noise=30, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

results = {}
# Vanilla GB (no regularization)
gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=4,
                                random_state=42).fit(X_tr, y_tr)
results['Vanilla GB'] = (gb.score(X_tr, y_tr), gb.score(X_te, y_te))

# XGBoost variants
for lam, gamma, label in [(0.0, 0.0, 'XGB (no reg)'),
                           (1.0, 0.0, 'XGB (λ=1)'),
                           (5.0, 1.0, 'XGB (λ=5, γ=1)')]:
    xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=4,
                        lambda_=lam, gamma=gamma, random_state=42).fit(X_tr, y_tr)
    results[label] = (xgb.score(X_tr, y_tr), xgb.score(X_te, y_te))

print(f"{'Model':<22} {'Train R²':>10} {'Test R²':>10} {'Overfit Gap':>12}")
print('-' * 56)
for name, (tr, te) in results.items():
    print(f"{name:<22} {tr:>10.4f} {te:>10.4f} {tr-te:>12.4f}")


## 4. min_child_weight — Hessian-Based Pruning

`min_child_weight` requires a minimum sum of hessians in each leaf.
For logistic loss where h = p(1-p), this effectively requires a minimum
number of uncertain (near-boundary) samples in each leaf.
This is more principled than `min_samples_leaf` in vanilla trees.


In [ ]:
X2, y2 = make_classification(n_samples=400, n_features=15, random_state=42)
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X2, y2, test_size=0.2, random_state=42)

mcw_values = [0.1, 1, 5, 10, 20]
train_accs, test_accs = [], []
for mcw in mcw_values:
    xgb = XGBClassifier(n_estimators=50, learning_rate=0.1, max_depth=5,
                         min_child_weight=mcw, random_state=42).fit(X_tr2, y_tr2)
    train_accs.append(xgb.score(X_tr2, y_tr2))
    test_accs.append(xgb.score(X_te2, y_te2))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(mcw_values, train_accs, 'o-', label='Train', color='#4C72B0', linewidth=2)
ax.plot(mcw_values, test_accs,  'o-', label='Test',  color='#DD8452', linewidth=2)
ax.set_xlabel('min_child_weight'); ax.set_ylabel('Accuracy')
ax.set_title('min_child_weight: Hessian-Based Regularization', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()
